In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, precision_recall_curve, auc
import joblib

In [20]:
print("1. Loading graph features...")
df = pd.read_csv('C:/Users/20aya/my_project/data/engineered_graph_features.csv')

1. Loading graph features...


In [21]:
# STEP 2: PRE-SPLIT FEATURE ENGINEERING
# ==========================================
print("2. Engineering Temporal and Velocity features...")
# Force chronological order and clean index
df = df.sort_values('TransactionDT').reset_index(drop=True)

# 2A. Time Since Last Txn
df['previous_txn_time'] = df.groupby('card1')['TransactionDT'].shift(1)
df['time_since_last_txn'] = (df['TransactionDT'] - df['previous_txn_time']).fillna(-999)
df = df.drop(columns=['previous_txn_time'])

# 2B. Rolling Velocity (Bulletproof Index Bypass)
df['dummy_datetime'] = pd.to_datetime(df['TransactionDT'], unit='s')

card_counts = df.groupby('card1').rolling('24h', on='dummy_datetime')['TransactionID'].count()
df['Card_Txn_Count_24h'] = card_counts.reset_index(level=0, drop=True).sort_index().values - 1

device_counts = df.groupby('DeviceInfo').rolling('1h', on='dummy_datetime')['TransactionID'].count()
df['Device_Txn_Count_1h'] = device_counts.reset_index(level=0, drop=True).sort_index().values - 1

df['Card_Txn_Count_24h'] = np.maximum(0, df['Card_Txn_Count_24h']).fillna(0).astype(int)
df['Device_Txn_Count_1h'] = np.maximum(0, df['Device_Txn_Count_1h']).fillna(0).astype(int)
df = df.drop(columns=['dummy_datetime'])

# 2C. Spend Ratios
df['historical_mean_amt'] = df.groupby('card1')['TransactionAmt'].transform(lambda x: x.shift().expanding().mean())
df['historical_mean_amt'] = df['historical_mean_amt'].fillna(df['TransactionAmt'])
df['spend_ratio'] = df['TransactionAmt'] / df['historical_mean_amt']
df = df.drop(columns=['historical_mean_amt'])



2. Engineering Temporal and Velocity features...


In [22]:
# STEP 3: THE CHRONOLOGICAL SPLIT
# ==========================================
print("3. Executing Train/Test Split...")
# Define all our shiny new features
features = [
    'TransactionAmt', 
    'DeviceInfo', # String (Categorical)
    'card_degree', 
    'device_degree', 
    'network_cluster_size',
    'time_since_last_txn',
    'Card_Txn_Count_24h',
    'Device_Txn_Count_1h',
    'spend_ratio'
]

X = df[features].copy()
y = df['isFraud']

split_idx = int(len(df) * 0.8)
X_train = X.iloc[:split_idx].copy() # .copy() prevents Pandas warnings later
y_train = y.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_test = y.iloc[split_idx:].copy()

# ==========================================
# STEP 4: POST-SPLIT TARGET ENCODING
# ==========================================
print("4. Applying Target Encoding (Leak-Free)...")
global_mean = y_train.mean()
smoothing_weight = 10
target_encode_maps = {}

# Learn from Train ONLY
stats = pd.DataFrame({'target': y_train, 'feature': X_train['DeviceInfo']})
grouped = stats.groupby('feature')['target'].agg(['count', 'mean'])
counts = grouped['count']
means = grouped['mean']
smoothed_vals = (counts * means + smoothing_weight * global_mean) / (counts + smoothing_weight)
target_encode_maps['DeviceInfo'] = smoothed_vals.to_dict()

# Map to Train and Test
X_train['DeviceInfo_risk_score'] = X_train['DeviceInfo'].map(lambda x: target_encode_maps['DeviceInfo'].get(x, global_mean))
X_test['DeviceInfo_risk_score'] = X_test['DeviceInfo'].map(lambda x: target_encode_maps['DeviceInfo'].get(x, global_mean))

# Drop original string column
X_train = X_train.drop(columns=['DeviceInfo'])
X_test = X_test.drop(columns=['DeviceInfo'])

# Fill remaining NaNs with 0
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)


## step 5
print("5. Training Optimized XGBoost Model...")

xgb_model = XGBClassifier(
    scale_pos_weight=15,          # Reduced from full imbalance ratio to prevent precision crash
    n_estimators=250,             # More trees...
    learning_rate=0.05,           # ...learning at a slower, more careful pace
    max_depth=6,                  # Slightly deeper to catch complex fraud rings
    subsample=0.8,                # Prevent overfitting to majority class
    colsample_bytree=0.8,         # Force the model to look at all features
    gamma=2,                      # Prune weak branches
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print("Model trained successfully!")

3. Executing Train/Test Split...
4. Applying Target Encoding (Leak-Free)...
5. Training Optimized XGBoost Model...
Model trained successfully!


In [23]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

print("=== DECISION THRESHOLD TUNING ===")
# Grab the raw probabilities from your already-trained model
probabilities = xgb_model.predict_proba(X_test)[:, 1]

# Test thresholds from 10% to 95%
thresholds = np.arange(0.1, 1.0, 0.05)

print(f"{'Threshold':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score'}")
print("-" * 55)

for thresh in thresholds:
    # Convert probabilities to hard 1s and 0s based on the new threshold
    test_preds = (probabilities >= thresh).astype(int)
    
    # Calculate metrics
    p = precision_score(y_test, test_preds, zero_division=0)
    r = recall_score(y_test, test_preds, zero_division=0)
    f1 = f1_score(y_test, test_preds, zero_division=0)
    
    print(f"{thresh:<10.2f} | {p:<10.3f} | {r:<10.3f} | {f1:.3f}")

=== DECISION THRESHOLD TUNING ===
Threshold  | Precision  | Recall     | F1-Score
-------------------------------------------------------
0.10       | 0.056      | 0.647      | 0.103
0.15       | 0.071      | 0.582      | 0.127
0.20       | 0.093      | 0.556      | 0.160
0.25       | 0.109      | 0.497      | 0.179
0.30       | 0.123      | 0.418      | 0.190
0.35       | 0.153      | 0.412      | 0.223
0.40       | 0.190      | 0.412      | 0.260
0.45       | 0.214      | 0.366      | 0.270
0.50       | 0.241      | 0.320      | 0.275
0.55       | 0.298      | 0.294      | 0.296
0.60       | 0.355      | 0.281      | 0.314
0.65       | 0.381      | 0.242      | 0.296
0.70       | 0.403      | 0.203      | 0.270
0.75       | 0.377      | 0.150      | 0.215
0.80       | 0.465      | 0.131      | 0.204
0.85       | 0.459      | 0.111      | 0.179
0.90       | 0.467      | 0.092      | 0.153
0.95       | 0.579      | 0.072      | 0.128


In [24]:
from sklearn.metrics import precision_recall_curve, auc

print("=== INITIATING FEATURE PRUNING ===")

# 1. Drop the noisy feature from both train and test sets
drop_cols = ['time_since_last_txn']
X_train_pruned = X_train.drop(columns=drop_cols)
X_test_pruned = X_test.drop(columns=drop_cols)

print(f"Dropped {drop_cols}. Remaining features: {len(X_train_pruned.columns)}")

# 2. Retrain the optimized model on the clean dataset
print("Retraining XGBoost without noise...")
xgb_model.fit(X_train_pruned, y_train)

# 3. Calculate the new PR-AUC
y_pred_proba_new = xgb_model.predict_proba(X_test_pruned)[:, 1]
precision_curve_new, recall_curve_new, _ = precision_recall_curve(y_test, y_pred_proba_new)
new_pr_auc = auc(recall_curve_new, precision_curve_new)

print(f"\n🏆 NEW PR-AUC SCORE: {new_pr_auc:.4f}")

=== INITIATING FEATURE PRUNING ===
Dropped ['time_since_last_txn']. Remaining features: 8
Retraining XGBoost without noise...

🏆 NEW PR-AUC SCORE: 0.2015


In [25]:
pip install seaborn


Note: you may need to restart the kernel to use updated packages.
